# Sprint 12 — APIs REST, mètodes HTTP i Pandas

En aquest notebook aplicarem el que hem après sobre **APIs REST**, **mètodes HTTP** i **codis d'estat**, fent servir la llibreria `requests` de Python per consultar dades d'APIs públiques, filtrar-les i desar-les en un `DataFrame` de pandas.

Estructura:
- **Nivell 1** — Exploració bàsica amb JSONPlaceholder (API de laboratori)
- **Nivell 2** — Interacció amb una API pública real (Open-Meteo)
- **Nivell 3** — API d'Open Data BCN (CKAN)


In [1]:
# Llibreries necessàries
import requests
import pandas as pd

pd.set_option("display.max_columns", None)


---
## Nivell 1 — JSONPlaceholder

API de laboratori: https://jsonplaceholder.typicode.com/


### 1 i 2. GET a `/posts`, `/users` i `/todos`
Obtenim els tres recursos i mostrem la quantitat total i el codi d'estat de cada petició.

In [2]:
BASE_URL = "https://jsonplaceholder.typicode.com"

recursos = ["posts", "users", "todos"]
dades = {}

for recurs in recursos:
    resposta = requests.get(f"{BASE_URL}/{recurs}")
    dades[recurs] = resposta.json()
    print(f"Recurs: {recurs}")
    print(f"  Codi d'estat: {resposta.status_code}")
    print(f"  Total d'elements: {len(dades[recurs])}")
    print("-" * 40)


Recurs: posts
  Codi d'estat: 200
  Total d'elements: 100
----------------------------------------
Recurs: users
  Codi d'estat: 200
  Total d'elements: 10
----------------------------------------
Recurs: todos
  Codi d'estat: 200
  Total d'elements: 200
----------------------------------------


### 3. Petició a una publicació inexistent (error 404)

In [3]:
resposta_404 = requests.get(f"{BASE_URL}/posts/99999")
print(f"Codi d'estat rebut: {resposta_404.status_code}")
print(f"Contingut de la resposta: {resposta_404.text}")


Codi d'estat rebut: 404
Contingut de la resposta: {}


### 4. POST — Crear una nova publicació fictícia

In [4]:
nova_publicacio = {
    "title": "El meu primer post amb requests",
    "body": "Aquest és el cos de la publicació de prova creada des de Python.",
    "userId": 1
}

resposta_post = requests.post(f"{BASE_URL}/posts", json=nova_publicacio)

print(f"Codi d'estat: {resposta_post.status_code}")
print("Resposta JSON:")
print(resposta_post.json())


Codi d'estat: 201
Resposta JSON:
{'title': 'El meu primer post amb requests', 'body': 'Aquest és el cos de la publicació de prova creada des de Python.', 'userId': 1, 'id': 101}


### 5. PATCH — Modificar parcialment una publicació existent

In [5]:
canvi_parcial = {
    "title": "Títol actualitzat amb PATCH"
}

resposta_patch = requests.patch(f"{BASE_URL}/posts/1", json=canvi_parcial)

print(f"Codi d'estat: {resposta_patch.status_code}")
print("Resposta JSON:")
print(resposta_patch.json())


Codi d'estat: 200
Resposta JSON:
{'userId': 1, 'id': 1, 'title': 'Títol actualitzat amb PATCH', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


### 6. DELETE — Eliminar una publicació

In [6]:
resposta_delete = requests.delete(f"{BASE_URL}/posts/1")

print(f"Codi d'estat: {resposta_delete.status_code}")
print("Resposta JSON:")
print(resposta_delete.json())


Codi d'estat: 200
Resposta JSON:
{}


---
## Nivell 2 — API pública real: Open-Meteo

He escollit **Open-Meteo** (https://open-meteo.com/), disponible al repositori [public-apis](https://github.com/public-apis/public-apis) dins la categoria *Weather*. 

### 3. Petició GET senzilla

In [7]:
url_forecast = "https://api.open-meteo.com/v1/forecast"

parametres = {
    "latitude": 41.39,     # Barcelona
    "longitude": 2.16,
    "current_weather": True,
    "hourly": "temperature_2m,precipitation",
    "timezone": "Europe/Madrid"
}

resposta_meteo = requests.get(url_forecast, params=parametres)

print(f"Codi d'estat: {resposta_meteo.status_code}")

dades_meteo = resposta_meteo.json()
print("Temps actual:", dades_meteo["current_weather"])
print("Primeres hores previstes:", dades_meteo["hourly"]["time"][:5])
print("Primeres temperatures previstes:", dades_meteo["hourly"]["temperature_2m"][:5])


Codi d'estat: 200
Temps actual: {'time': '2026-07-06T22:00', 'interval': 900, 'temperature': 27.0, 'windspeed': 2.2, 'winddirection': 180, 'is_day': 0, 'weathercode': 0}
Primeres hores previstes: ['2026-07-06T00:00', '2026-07-06T01:00', '2026-07-06T02:00', '2026-07-06T03:00', '2026-07-06T04:00']
Primeres temperatures previstes: [27.2, 26.7, 26.1, 25.7, 25.4]


### 4. Conversió a DataFrame de pandas

In [8]:
df_meteo = pd.DataFrame({
    "data_hora": dades_meteo["hourly"]["time"],
    "temperatura_C": dades_meteo["hourly"]["temperature_2m"],
    "precipitacio_mm": dades_meteo["hourly"]["precipitation"]
})

df_meteo.head()


,data_hora,temperatura_C,precipitacio_mm
0,2026-07-06T00:00,27.2,0.0
1,2026-07-06T01:00,26.7,0.0
2,2026-07-06T02:00,26.1,0.0
3,2026-07-06T03:00,25.7,0.0
4,2026-07-06T04:00,25.4,0.0


---
## Nivell 3 — Open Data BCN

Utilitzarem l'API CKAN d'Open Data BCN: https://opendata-ajuntament.barcelona.cat/

Dataset escollit: **"Noms dels habitants de Barcelona per edat mitjana i sexe"**


In [9]:
BASE_CKAN = "https://opendata-ajuntament.barcelona.cat/data/api/action"
SLUG_CONEGUT = "pad_m_nom_sexe"  # "Noms dels habitants de Barcelona per edat mitjana i sexe"

# 1. Cerquem el dataset amb package_search
# Nota: la cerca de text complet de CKAN és sensible a com estan indexades les paraules;
# val més fer servir termes senzills i genèrics que una frase llarga i específica.
resposta_search = requests.get(
    f"{BASE_CKAN}/package_search",
    params={"q": "noms habitants sexe"}
)

print(f"Codi d'estat package_search: {resposta_search.status_code}")

resultats_search = resposta_search.json()["result"]["results"]
print(f"Nombre de resultats trobats: {len(resultats_search)}")
for paquet in resultats_search[:5]:
    print("-", paquet["name"], "|", paquet["title"])

# Si la cerca de text no troba res, fem servir el slug real ja conegut com a alternativa
if not resultats_search:
    print(f"\nCap resultat amb aquesta cerca; farem servir el dataset conegut: {SLUG_CONEGUT}")


Codi d'estat package_search: 200
Nombre de resultats trobats: 10
- pad_mdb_nacionalitat-regio_sexe | Population by geographical region of nationality and sex
- pad_mdb_lloc-naix-regio_sexe | Population by geographical region of birth and sex
- pad_m_nom_sexe | Names of the inhabitants of Barcelona by average age and sex
- pad_m_cognom | Surnames of the inhabitants of Barcelona
- poblacio_projectada | Projected population by scenario and sex


In [10]:
# 2. Detall del dataset amb package_show
# IMPORTANT: no endevinem el nom del dataset a mà (el catàleg canvia i els slugs
# no sempre coincideixen amb el títol). Agafem el 'name' real del primer resultat
# que ens ha tornat package_search a la cel·la anterior; si no hi ha resultats,
# fem servir el slug conegut (SLUG_CONEGUT) com a alternativa.
nom_dataset = resultats_search[0]["name"] if resultats_search else SLUG_CONEGUT
print(f"Dataset seleccionat: {nom_dataset}")

resposta_show = requests.get(
    f"{BASE_CKAN}/package_show",
    params={"id": nom_dataset}
)

print(f"Codi d'estat package_show: {resposta_show.status_code}")

detall_dataset = resposta_show.json()["result"]

for recurs in detall_dataset["resources"]:
    print(f"- {recurs['name']} | format: {recurs['format']} | resource_id: {recurs['id']}")


Dataset seleccionat: pad_mdb_nacionalitat-regio_sexe
Codi d'estat package_show: 200
- 2026_pad_mdb_nacionalitat-regio_sexe.csv | format: CSV | resource_id: 9577a42e-224d-42ea-ae47-039824b265df
- 2026_pad_mdb_nacionalitat-regio_sexe.json | format: JSON | resource_id: 58009cb4-9623-4a29-a38b-e913a562f623
- 2025_pad_mdb_nacionalitat-regio_sexe.csv | format: CSV | resource_id: 3dce0dbd-5a7d-439a-b945-6bffc17a6c03
- 2025_pad_mdb_nacionalitat-regio_sexe.json | format: JSON | resource_id: 0dea1610-e615-4d81-9714-a374e1bb74c9
- 2024_pad_mdb_nacionalitat-regio_sexe.csv | format: CSV | resource_id: 8caae93c-6196-490f-abd2-87eccda80776
- 2024_pad_mdb_nacionalitat-regio_sexe.json | format: JSON | resource_id: 08e3a0d7-7758-4d76-9e4b-89f2c2dc14c3
- 2023_pad_mdb_nacionalitat-regio_sexe.csv | format: CSV | resource_id: 7b7efabe-f08f-49e0-b1f9-eb116c6fdcc9
- 2023_pad_mdb_nacionalitat-regio_sexe.json | format: JSON | resource_id: 026f6408-2fbe-404d-9620-6198bea2fd87
- 2022_pad_mdb_nacionalitat-regio_se

In [11]:
# 3. Consulta amb datastore_search sobre un recurs concret (mínim 100 registres)
# Agafem el resource_id d'un recurs real del dataset (per exemple, el primer que trobem).
# Si un recurs concret no està donat d'alta al DataStore, datastore_search tornarà un
# error clar ("Resource ... not found"); en aquest cas, prova amb un altre recurs de la llista.
resource_id = detall_dataset["resources"][0]["id"]
print(f"resource_id seleccionat: {resource_id}")

resposta_datastore = requests.get(
    f"{BASE_CKAN}/datastore_search",
    params={"resource_id": resource_id, "limit": 100}
)

print(f"Codi d'estat datastore_search: {resposta_datastore.status_code}")

resultat_datastore = resposta_datastore.json()["result"]
registres = resultat_datastore["records"]

print(f"Nombre de registres obtinguts: {len(registres)}")
registres[:3]


resource_id seleccionat: 9577a42e-224d-42ea-ae47-039824b265df
Codi d'estat datastore_search: 200
Nombre de registres obtinguts: 100


[{'NACIONALITAT_REGIO': '1',
  'Codi_Districte': '1',
  'SEXE': '1',
  'Nom_Districte': 'Ciutat Vella',
  'Codi_Barri': '1',
  'Nom_Barri': 'el Raval',
  'Valor': '17',
  'Data_Referencia': '2026-01-01T00:00:00',
  '_id': 1},
 {'NACIONALITAT_REGIO': '1',
  'Codi_Districte': '1',
  'SEXE': '2',
  'Nom_Districte': 'Ciutat Vella',
  'Codi_Barri': '1',
  'Nom_Barri': 'el Raval',
  'Valor': '30',
  'Data_Referencia': '2026-01-01T00:00:00',
  '_id': 2},
 {'NACIONALITAT_REGIO': '2',
  'Codi_Districte': '1',
  'SEXE': '1',
  'Nom_Districte': 'Ciutat Vella',
  'Codi_Barri': '1',
  'Nom_Barri': 'el Raval',
  'Valor': '13',
  'Data_Referencia': '2026-01-01T00:00:00',
  '_id': 3}]

### 4. Conversió a DataFrame

In [12]:
df_bcn = pd.DataFrame(registres)
df_bcn.head()


,NACIONALITAT_REGIO,Codi_Districte,SEXE,Nom_Districte,Codi_Barri,Nom_Barri,Valor,Data_Referencia,_id
0,1,1,1,Ciutat Vella,1,el Raval,17,2026-01-01T00:00:00,1
1,1,1,2,Ciutat Vella,1,el Raval,30,2026-01-01T00:00:00,2
2,2,1,1,Ciutat Vella,1,el Raval,13,2026-01-01T00:00:00,3
3,2,1,2,Ciutat Vella,1,el Raval,19,2026-01-01T00:00:00,4
4,3,1,1,Ciutat Vella,1,el Raval,668,2026-01-01T00:00:00,5


### 5. Desar el DataFrame en un fitxer .csv

In [13]:
df_bcn.to_csv("opendata_bcn_noms_frequents.csv", index=False, encoding="utf-8")
print("Fitxer 'opendata_bcn_noms_frequents.csv' desat correctament.")


Fitxer 'opendata_bcn_noms_frequents.csv' desat correctament.
